In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve().parent
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"

tables = sorted(TABLES.glob("*.csv"))
figures = sorted(FIGURES.glob("*.png"))

inventory = pd.DataFrame(
    [(p.stem, p.suffix, "Table") for p in tables] +
    [(p.stem, p.suffix, "Figure") for p in figures],
    columns=["Filename", "Format", "Type"]
)

print(f"Tables: {len(tables)}")
print(f"Figures: {len(figures)}")
print("\nAvailable outputs:")
print(inventory.to_string(index=False))

Tables: 20
Figures: 7

Available outputs:
                                    Filename Format   Type
                inequality_contrasts_numeric   .csv  Table
                    m0_geographic_regression   .csv  Table
                 m1_socioeconomic_regression   .csv  Table
              m2_average_marginal_effects_ci   .csv  Table
           m2_average_marginal_effects_point   .csv  Table
                m2_fully_adjusted_regression   .csv  Table
                        m2_model_diagnostics   .csv  Table
     m2_standardised_income_probabilities_ci   .csv  Table
  m2_standardised_income_probabilities_point   .csv  Table
                        nested_model_summary   .csv  Table
                    regression_public_sector   .csv  Table
                      regression_travel_time   .csv  Table
                       T17_interaction_tests   .csv  Table
    T18_intersectional_probability_contrasts   .csv  Table
                  T19_vulnerability_profiles   .csv  Table
              

In [2]:
# Final outputs required for the manuscript
required_tables = [
    "T17_interaction_tests.csv",
    "T19_vulnerability_profiles.csv",
    "T20_profile_population_burden.csv",
    "m2_fully_adjusted_regression.csv",
    "m2_average_marginal_effects_ci.csv",
    "table_inequality_contrasts.csv"
]

required_figures = [
    "F11_vulnerability_gradient",
    "F12_population_disadvantage_profiles"
]

# Check file availability
for name in required_tables:
    print(f"{'PASS' if (TABLES / name).exists() else 'MISSING'}: {name}")

for name in required_figures:
    png = (FIGURES / f"{name}.png").exists()
    pdf = (FIGURES / f"{name}.pdf").exists()
    print(f"{'PASS' if png and pdf else 'CHECK'}: {name} (PNG={png}, PDF={pdf})")

# Validate vulnerability estimates
t19 = pd.read_csv(TABLES / "T19_vulnerability_profiles.csv")
t20 = pd.read_csv(TABLES / "T20_profile_population_burden.csv")

print("\nVulnerability checks:")
print("Population share:", round(t19["Household_share_pct"].sum(), 2), "%")
print("Travel burden share:", round(t20["Share_of_travel_burden_pct"].sum(), 2), "%")
print("Valid confidence intervals:",
      ((t19["CI_lower_pct"] <= t19["Travel_30min_pct"]) &
       (t19["Travel_30min_pct"] <= t19["CI_upper_pct"])).all())

print("\nEXCLUDED: T18_intersectional_probability_contrasts.csv")

PASS: T17_interaction_tests.csv
PASS: T19_vulnerability_profiles.csv
PASS: T20_profile_population_burden.csv
PASS: m2_fully_adjusted_regression.csv
PASS: m2_average_marginal_effects_ci.csv
PASS: table_inequality_contrasts.csv
PASS: F11_vulnerability_gradient (PNG=True, PDF=True)
PASS: F12_population_disadvantage_profiles (PNG=True, PDF=True)

Vulnerability checks:
Population share: 100.0 %
Travel burden share: 100.0 %
Valid confidence intervals: True

EXCLUDED: T18_intersectional_probability_contrasts.csv


In [3]:
# Load final analytical tables
t17 = pd.read_csv(TABLES / "T17_interaction_tests.csv")
t19 = pd.read_csv(TABLES / "T19_vulnerability_profiles.csv")
t20 = pd.read_csv(TABLES / "T20_profile_population_burden.csv")
ame = pd.read_csv(TABLES / "m2_average_marginal_effects_ci.csv")
inequality = pd.read_csv(TABLES / "table_inequality_contrasts.csv")

# RQ1: Descriptive inequalities
print("RQ1 — DESCRIPTIVE INEQUALITIES")
print("Available columns:", inequality.columns.tolist())
print(inequality.head(8).to_string(index=False))

# RQ2: Adjusted associations
print("\nRQ2 — ADJUSTED ASSOCIATIONS")
print("Available columns:", ame.columns.tolist())
print(ame.head(8).to_string(index=False))

# RQ3: Interactions and vulnerability
print("\nRQ3 — INTERACTION TESTS")
print(t17.to_string(index=False))

print("\nRQ3 — VULNERABILITY PROFILES")
print(t19.round(2).to_string(index=False))

print("\nRQ3 — POPULATION BURDEN")
print(t20.round(2).to_string(index=False))

RQ1 — DESCRIPTIVE INEQUALITIES
Available columns: ['Indicator', 'Outcome', 'Comparison', 'Reference', 'Difference, pp (95% CI)', 'Prevalence ratio (95% CI)']
           Indicator                      Outcome          Comparison            Reference Difference, pp (95% CI) Prevalence ratio (95% CI)
 Geographic location      Travel time ≥30 minutes           Non-metro                Metro     15.63 (14.22–17.04)          3.61 (3.16–4.11)
 Geographic location Usual public-sector facility           Non-metro                Metro     16.65 (15.00–18.31)          1.26 (1.23–1.29)
    Household income      Travel time ≥30 minutes  Q1 (Lowest income)  Q5 (Highest income)     15.31 (13.39–17.22)          3.00 (2.57–3.49)
    Household income Usual public-sector facility  Q1 (Lowest income)  Q5 (Highest income)     68.40 (66.42–70.39)          3.59 (3.36–3.85)
     Food insecurity      Travel time ≥30 minutes Any food insecurity   No food insecurity       9.99 (8.39–11.58)          1.86 (1.70–2.

In [4]:
publication = {
    "RQ1": [
        "table_inequality_contrasts.csv",
        "Provincial_Healthcare_Access_Inequalities.png",
        "income_healthcare_access_gradient.png"
    ],
    "RQ2": [
        "m2_fully_adjusted_regression.csv",
        "m2_average_marginal_effects_ci.csv"
    ],
    "RQ3": [
        "T17_interaction_tests.csv",
        "T19_vulnerability_profiles.csv",
        "T20_profile_population_burden.csv",
        "F11_vulnerability_gradient.png",
        "F12_population_disadvantage_profiles.png"
    ]
}

records = []

for rq, filenames in publication.items():
    for name in filenames:
        folder = FIGURES if name.endswith(".png") else TABLES
        path = folder / name
        records.append({
            "RQ": rq,
            "File": name,
            "Status": "PASS" if path.is_file() and path.stat().st_size > 0 else "MISSING"
        })

checklist = pd.DataFrame(records)
print(checklist.to_string(index=False))
print("\nMissing files:", (checklist["Status"] == "MISSING").sum())
print("Excluded: T18_intersectional_probability_contrasts.csv")

 RQ                                          File Status
RQ1                table_inequality_contrasts.csv   PASS
RQ1 Provincial_Healthcare_Access_Inequalities.png   PASS
RQ1         income_healthcare_access_gradient.png   PASS
RQ2              m2_fully_adjusted_regression.csv   PASS
RQ2            m2_average_marginal_effects_ci.csv   PASS
RQ3                     T17_interaction_tests.csv   PASS
RQ3                T19_vulnerability_profiles.csv   PASS
RQ3             T20_profile_population_burden.csv   PASS
RQ3                F11_vulnerability_gradient.png   PASS
RQ3      F12_population_disadvantage_profiles.png   PASS

Missing files: 0
Excluded: T18_intersectional_probability_contrasts.csv


In [5]:
synthesis = pd.DataFrame([
    {
        "RQ": "RQ1",
        "Focus": "Descriptive healthcare inequalities",
        "Key_finding": "Non-metro households have 15.63 pp higher prolonged-travel prevalence than metro households.",
        "Evidence": "table_inequality_contrasts.csv",
        "Status": "Answered"
    },
    {
        "RQ": "RQ2",
        "Focus": "Adjusted socioeconomic and geographic associations",
        "Key_finding": "Non-metro residence (+4.66 pp) and food insecurity (+5.36 pp) remain associated with prolonged travel.",
        "Evidence": "m2_average_marginal_effects_ci.csv",
        "Status": "Answered"
    },
    {
        "RQ": "RQ3",
        "Focus": "Intersectional vulnerability and population burden",
        "Key_finding": "Households with three disadvantages have 32.62% prolonged-travel prevalence; those with two or three account for 61.74% of the burden.",
        "Evidence": "T17_interaction_tests.csv; T19_vulnerability_profiles.csv; T20_profile_population_burden.csv",
        "Status": "Answered"
    }
])

output = TABLES / "final_research_question_synthesis.csv"
synthesis.to_csv(output, index=False)

assert output.exists()
assert len(synthesis) == 3
assert (synthesis["Status"] == "Answered").all()

print(synthesis[["RQ", "Focus", "Status"]].to_string(index=False))
print("\nSaved:", output.name)
print("Research questions addressed:", len(synthesis), "of 3")
print("Publication outputs verified:", len(checklist))

 RQ                                              Focus   Status
RQ1                Descriptive healthcare inequalities Answered
RQ2 Adjusted socioeconomic and geographic associations Answered
RQ3 Intersectional vulnerability and population burden Answered

Saved: final_research_question_synthesis.csv
Research questions addressed: 3 of 3
Publication outputs verified: 10
